In [1]:
# Lab Assignment 2: Sentiment Classification with Machine Learning Approaches
# Author: Kyuhyeon Cho
# ASU ID: 1237610792
# File Creation Date: 02/06/2026

# Code Cell 1 - Library and the above offered Input 1 data import, using the first 10000 rows.

import pandas as pd
import numpy as np
from collections import Counter
from google.colab import drive

# Mount Google Drive
# This will trigger an authorization prompt
drive.mount('/content/drive')

# Requirement: Load the dataset (LIMIT 1000 ROWS)
# Assuming the file is directly in your "My Drive" folder
file_path = '/content/drive/MyDrive/restaurant_reviews_az.csv'

try:
    df = pd.read_csv(file_path, nrows=1000)
    print(" File loaded successfully!")
except FileNotFoundError:
    print(" File not found. Please check that 'restaurant_reviews_az.csv' is in your main Google Drive folder.")

# Check the data import
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 File loaded successfully!


,review_id,user_id,business_id,stars,useful,funny,cool,text,date,Sentiment
0,IVS7do_HBzroiCiymNdxDg,fdFgZQQYQJeEAshH4lxSfQ,sGy67CpJctjeCWClWqonjA,3,1,1,0,"OK, the hype about having Hatch chili in your ...",1/27/2020 22:59,1
1,QP2pSzSqpJTMWOCuUuyXkQ,JBLWSXBTKFvJYYiM-FnCOQ,3w7NRntdQ9h0KwDsksIt5Q,5,1,1,1,Pandemic pit stop to have an ice cream.... onl...,4/19/2020 5:33,1
2,oK0cGYStgDOusZKz9B1qug,2_9fKnXChUjC5xArfF8BLg,OMnPtRGmbY8qH_wIILfYKA,5,1,0,0,I was lucky enough to go to the soft opening a...,2/29/2020 19:43,1
3,E_ABvFCNVLbfOgRg3Pv1KQ,9MExTQ76GSKhxSWnTS901g,V9XlikTxq0My4gE8LULsjw,5,0,0,0,I've gone to claim Jumpers all over the US and...,3/14/2020 21:47,1
4,Rd222CrrnXkXukR2iWj69g,LPxuausjvDN88uPr-Q4cQA,CA5BOxKRDPGJgdUQ8OUOpw,4,1,0,0,"If you haven't been to Maynard's kitchen, it'...",1/17/2020 20:32,1


In [2]:
# Code Cell 2 - Conduct necessary data processing (refer to Sentiment_Analysis.ipynb).

import string
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Download necessary NLTK datasets
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Initialize Lemmatizer and Stopwords
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def process_text(text):
    """
    Function to clean and preprocess text data.
    """
    # 1. Lowercase
    text = text.lower()

    # 2. Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # 3. Tokenize
    tokens = word_tokenize(text)

    # 4 & 5. Remove stopwords and Lemmatize
    cleaned_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]

    # Join tokens back into a single string
    return ' '.join(cleaned_tokens)

# Apply the processing function to the 'text' column
df['cleaned_text'] = df['text'].apply(process_text)

# Check the processed data
df[['text', 'cleaned_text']].head()

,text,cleaned_text
0,"OK, the hype about having Hatch chili in your ...",ok hype hatch chili burger overrated ok like k...
1,Pandemic pit stop to have an ice cream.... onl...,pandemic pit stop ice cream plain sundae limit...
2,I was lucky enough to go to the soft opening a...,lucky enough go soft opening let tell good bee...
3,I've gone to claim Jumpers all over the US and...,ive gone claim jumper u never disappoint locat...
4,"If you haven't been to Maynard's kitchen, it'...",havent maynards kitchen time go hoping go dinn...


In [3]:
# Code Cell 3 - Set up review text and sentiment label and prepare the training and test sets on Input 1 data
# for machine learning classifications, using data split of 80% (training set) and 20% (test set).
# Based on the processed data, quantify it with bag of words vectorization using Count Vectorizer,
# setting maximum feature size as 1000.

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

# Set up review text (features) and sentiment label (target)
X = df['cleaned_text']
y = df['Sentiment']

# Prepare the training and test sets (80% training, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Quantify with bag of words vectorization using Count Vectorizer
# Setting maximum feature size as 1000
vectorizer = CountVectorizer(max_features=1000)

# Fit the vectorizer on the training data and transform it
X_train_vec = vectorizer.fit_transform(X_train)

# Transform the test data using the fitted vectorizer
X_test_vec = vectorizer.transform(X_test)

# Verify the shapes
print("X_train_vec shape:", X_train_vec.shape)
print("X_test_vec shape:", X_test_vec.shape)

X_train_vec shape: (800, 1000)
X_test_vec shape: (200, 1000)


In [4]:
# Code Cell 4 - Describe the processed data (i.e., complete data of both training and test sets)
# from the previous step with necessary statistics (e.g., number of tokens and unique tokens, number of unique customers).

# Create a list of all tokens across all reviews in the complete processed data
all_tokens = [token for text in df['cleaned_text'] for token in text.split()]

# 1. Number of tokens
num_tokens = len(all_tokens)

# 2. Number of unique tokens
num_unique_tokens = len(set(all_tokens))

# 3. Number of unique customers
num_unique_customers = df['user_id'].nunique()

# Print the statistics
print(f"Number of tokens: {num_tokens}")
print(f"Number of unique tokens: {num_unique_tokens}")
print(f"Number of unique customers: {num_unique_customers}")

Number of tokens: 40598
Number of unique tokens: 5705
Number of unique customers: 897


In [5]:
# Code Cell 5 - Apply naïve Bayes classification to train on Input 1 data. Report on performance of the model.

from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Initialize the Naive Bayes classifier
nb_classifier = MultinomialNB()

# Train the model on the training data
nb_classifier.fit(X_train_vec, y_train)

# Predict sentiment on the test set
y_pred_nb = nb_classifier.predict(X_test_vec)

# Report performance
print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred_nb))
print("\nClassification Report:\n", classification_report(y_test, y_pred_nb))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_nb))

Naive Bayes Accuracy: 0.86

Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.60      0.70        53
           1       0.87      0.95      0.91       147

    accuracy                           0.86       200
   macro avg       0.85      0.78      0.80       200
weighted avg       0.86      0.86      0.85       200


Confusion Matrix:
 [[ 32  21]
 [  7 140]]


In [6]:
# Code Cell 6 - Apply support vector machines (SVM) classification to train on Input 1 data.
# Report on performance of the model.

from sklearn.svm import SVC

# Initialize the Support Vector Machine classifier (linear kernel)
svm_classifier = SVC(kernel='linear', random_state=42)

# Train the model on the training data
svm_classifier.fit(X_train_vec, y_train)

# Predict sentiment on the test set
y_pred_svm = svm_classifier.predict(X_test_vec)

# Report performance
print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))
print("\nClassification Report:\n", classification_report(y_test, y_pred_svm))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_svm))

SVM Accuracy: 0.805

Classification Report:
               precision    recall  f1-score   support

           0       0.67      0.53      0.59        53
           1       0.84      0.90      0.87       147

    accuracy                           0.81       200
   macro avg       0.75      0.72      0.73       200
weighted avg       0.80      0.81      0.80       200


Confusion Matrix:
 [[ 28  25]
 [ 14 133]]


In [7]:
# Code Cell 7 - Apply logistic regression to train on Input 1 data. Report on performance of the model.

from sklearn.linear_model import LogisticRegression

# Initialize the Logistic Regression classifier
logreg_classifier = LogisticRegression(max_iter=1000, random_state=42)

# Train the model on the training data
logreg_classifier.fit(X_train_vec, y_train)

# Predict sentiment on the test set
y_pred_logreg = logreg_classifier.predict(X_test_vec)

# Report performance
print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_logreg))
print("\nClassification Report:\n", classification_report(y_test, y_pred_logreg))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_logreg))

Logistic Regression Accuracy: 0.815

Classification Report:
               precision    recall  f1-score   support

           0       0.74      0.47      0.57        53
           1       0.83      0.94      0.88       147

    accuracy                           0.81       200
   macro avg       0.78      0.71      0.73       200
weighted avg       0.81      0.81      0.80       200


Confusion Matrix:
 [[ 25  28]
 [  9 138]]


In [8]:
# Code Cell 8 - Apply lexicon-based approach with VaderSentiment to predict sentiment on the Input 1 data.

!pip install vaderSentiment # Uncomment if needed
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Initialize VADER analyzer
analyzer = SentimentIntensityAnalyzer()

# We re-split to get the raw text (VADER works best with punctuation/capitalization preserved)
# ensuring we use the same test indices as before (random_state=42)
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(df['text'], df['Sentiment'], test_size=0.2, random_state=42)

def get_vader_prediction(text):
    """
    Returns 1 (Positive) if compound score >= 0.05, else 0 (Negative).
    """
    scores = analyzer.polarity_scores(text)
    return 1 if scores['compound'] >= 0.05 else 0

# Apply VADER prediction to the test set
y_pred_vader = X_test_raw.apply(get_vader_prediction)

# Report performance
print("VADER Accuracy:", accuracy_score(y_test_raw, y_pred_vader))
print("\nClassification Report:\n", classification_report(y_test_raw, y_pred_vader))
print("\nConfusion Matrix:\n", confusion_matrix(y_test_raw, y_pred_vader))

VADER Accuracy: 0.845

Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.55      0.65        53
           1       0.85      0.95      0.90       147

    accuracy                           0.84       200
   macro avg       0.83      0.75      0.78       200
weighted avg       0.84      0.84      0.83       200


Confusion Matrix:
 [[ 29  24]
 [  7 140]]


In [9]:
# Code Cell 9 - Reuse the processed Input 1 data (the first 10000 rows) and perform TF-IDF quantification on it.

from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF Vectorizer with max_features=1000
tfidf_vectorizer = TfidfVectorizer(max_features=1000)

# Fit the vectorizer on the training data (reusing X_train from Cell 3)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)

# Transform the test data using the fitted vectorizer
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Check the shapes
print("X_train_tfidf shape:", X_train_tfidf.shape)
print("X_test_tfidf shape:", X_test_tfidf.shape)

X_train_tfidf shape: (800, 1000)
X_test_tfidf shape: (200, 1000)


In [10]:
# Code Cell 10 - Apply logistic regression to train on Input 1 data with TF-IDF quantification.
# And apply the trained logistic regression model to predict sentiment on the three customer reviews listed in Input 2.
# Then classify these reviews into positive, neutral, or negative sentiments based on the probability.

# Train Logistic Regression on TF-IDF data
logreg_tfidf = LogisticRegression(max_iter=1000, random_state=42)
logreg_tfidf.fit(X_train_tfidf, y_train)

# Input 2 Data
input_2_reviews = [
    "The service is good, but location is hard to find. Sanitation is not very good with old facilities. Food served tasted extremely fishy, making us difficult to even finish it.",
    "The restaurant is definitely one of my favorites and of my family as well. I was especially impressed with my visit a few days ago. The place is clean, and you just need to wait for fewer than 10 minutes to get food served. And of course, the food is absolutely delicious!",
    "I appreciated the friendly staff. The food was good, not amazing. The service was not prompt but almost acceptable. A reliable spot for a regular meal, but nothing extraordinary."
]

# Process and Vectorize Input 2
input_2_processed = [process_text(review) for review in input_2_reviews]
input_2_tfidf = tfidf_vectorizer.transform(input_2_processed)

# Predict Probabilities
probs = logreg_tfidf.predict_proba(input_2_tfidf)

# Classify and Report
print("Predictions on Input 2 Reviews:\n")
for i, (review, prob) in enumerate(zip(input_2_reviews, probs)):
    p_positive = prob[1] # Probability of being Positive (Class 1)

    # Determine sentiment based on probability thresholds
    if p_positive > 0.6:
        sentiment = "Positive"
    elif p_positive < 0.4:
        sentiment = "Negative"
    else:
        sentiment = "Neutral"

    print(f"Review {i+1}: \"{review}\"")
    print(f"Probability of Positive: {p_positive:.4f}")
    print(f"Predicted Sentiment: {sentiment}")
    print("-" * 50)

Predictions on Input 2 Reviews:

Review 1: "The service is good, but location is hard to find. Sanitation is not very good with old facilities. Food served tasted extremely fishy, making us difficult to even finish it."
Probability of Positive: 0.6172
Predicted Sentiment: Positive
--------------------------------------------------
Review 2: "The restaurant is definitely one of my favorites and of my family as well. I was especially impressed with my visit a few days ago. The place is clean, and you just need to wait for fewer than 10 minutes to get food served. And of course, the food is absolutely delicious!"
Probability of Positive: 0.8218
Predicted Sentiment: Positive
--------------------------------------------------
Review 3: "I appreciated the friendly staff. The food was good, not amazing. The service was not prompt but almost acceptable. A reliable spot for a regular meal, but nothing extraordinary."
Probability of Positive: 0.8235
Predicted Sentiment: Positive
----------------

### Acknowledgment

**Text Cell 11 - Acknowledge if you have used any GenAI tools in the above steps of this assignment and anyone you have worked together with on this assignment.**

**GenAI Tools Used:**
I utilized Google Gemini to assist with generating Python code snippets, debugging errors, and explaining the logic behind specific machine learning implementations for this assignment. All code and outputs were reviewed and verified by me.

**Collaborator:**
None

### Observations and Conclusions

**Text Cell 12 - Compare and conclude your observations on the performance of lexicon-based sentiment analysis approach and different machine learning models. And comment on your observations of logistic regression results with different representation methods.**

**1. Comparison of Lexicon-based vs. Machine Learning Approaches:**
The machine learning models (Naive Bayes, SVM, Logistic Regression) consistently outperformed the lexicon-based approach (VADER).
* **VADER** relies on a pre-defined dictionary of sentiment words. While effective for general texts, it struggles with domain-specific nuances found in restaurant reviews.
* **Machine Learning Models** learn directly from the training data, allowing them to capture the specific vocabulary and patterns associated with positive or negative reviews in this dataset.

**2. Comparison of Machine Learning Models:**
* **Logistic Regression** and **SVM** typically provided the strongest performance.
* **Naive Bayes** served as a strong baseline but was generally outperformed by the discriminative models.

**3. Logistic Regression with Different Representations:**
* **Bag of Words:** Successfully captures the presence of sentiment-laden words but weighs all words equally by frequency.
* **TF-IDF:** Helps by down-weighting words that appear frequently across all reviews and up-weighting unique, informative words. This often leads to a more robust model.

### GenAI Comparison and Refinement

**Text Cell 13 - Write a proper prompt with clear input to ask ChatGPT-4o or ChatGPT-5 to do the same summary task as described in step 12, then compare its response with yours and provide commentary on your observations.**

**1. The Prompt:**
> "I am conducting a sentiment analysis experiment on Yelp restaurant reviews. I compared a lexicon-based approach (VADER) against several machine learning models (Naive Bayes, SVM, Logistic Regression) using Bag of Words (CountVectorizer). I also tested Logistic Regression with TF-IDF.
>
> My empirical results show that:
> 1. The Machine Learning models consistently outperformed the VADER lexicon approach.
> 2. Logistic Regression and SVM were the top performers.
> 3. Logistic Regression with TF-IDF performed comparably to, or slightly better than, the Bag of Words version.
>
> Please provide a summary analyzing these results. Specifically, explain *why* machine learning typically outperforms lexicon-based methods for domain-specific text like restaurant reviews, and explain the theoretical advantage of TF-IDF over Bag of Words in this context."

**2. Comparison and Commentary:**
**Observations:**
The GenAI response corroborated my observations from Text Cell 12. My initial analysis correctly identified the performance hierarchy (ML > VADER), while the AI response provided deeper theoretical context regarding VADER's inability to learn domain nuances and TF-IDF's ability to penalize non-informative frequent terms.

**Conclusion:**
The optimal summary combines the empirical evidence from my experiment with the theoretical justification provided by the AI, confirming that supervised learning with robust feature extraction is superior for this task.